# React 框架 · 第 08 天

## 极客头条：文章发布、封面上传与列表展示

**资料范围**：以 day08 PPT 的讲授顺序为主，结合配套总笔记中“发布文章模块”和“文章列表模块”前半部分，以及 `react-jike` 随堂代码。day08 的重点是完成文章发布表单、封面上传，再搭出文章列表和状态展示；筛选、分页、删除及编辑属于第 09 天。

## 一、本日学习路线

```text
发布页静态结构
  → 富文本编辑器与频道选项
  → Form 收集字段并提交文章
  → Upload 上传封面 / 单图三图无图模式
  → 文章列表筛选区与 Table 静态结构
  → 频道复用、接口列表与状态 Tag
```

完成后应能把标题、频道、内容和封面组装成接口请求体，并让文章列表从接口数据渲染出来。

## 二、发布页静态结构

页面由面包屑、Card、Form、Input、Select、富文本和提交按钮组成：

```jsx
<Card
  title={<Breadcrumb items={[
    { title: <Link to="/">首页</Link> },
    { title: '发布文章' },
  ]} />}
>
  <Form
    labelCol={{ span: 4 }}
    wrapperCol={{ span: 16 }}
    initialValues={{ type: 0 }}
    onFinish={onFinish}
  >
    <Form.Item name="title" label="标题" rules={[{ required: true }]}>
      <Input placeholder="请输入文章标题" />
    </Form.Item>
    <Form.Item name="channel_id" label="频道" rules={[{ required: true }]}>
      <Select placeholder="请选择文章频道">{/* 动态选项 */}</Select>
    </Form.Item>
    <Form.Item name="content" label="内容" rules={[{ required: true }]}>
      {/* 富文本编辑器 */}
    </Form.Item>
    <Button type="primary" htmlType="submit">发布文章</Button>
  </Form>
</Card>
```

先确定表单字段名与接口字段名一致，再接入真实数据。`Form.Item` 的 `name` 不只是标签，它决定表单提交对象的字段。

![发布页面结构](assets/day08/15.png)

## 三、富文本编辑器

课程使用 `react-quill`：

```bash
npm i react-quill@2.0.0-beta.2
```

```jsx
import ReactQuill from 'react-quill'
import 'react-quill/dist/quill.snow.css'

<Form.Item
  name="content"
  label="内容"
  rules={[{ required: true, message: '请输入文章内容' }]}
>
  <ReactQuill theme="snow" placeholder="请输入文章内容" />
</Form.Item>
```

编辑器生成的内容是 HTML 字符串；提交前可检查是否只有空标签，显示已发布内容时应遵守项目的可信内容处理规则。课程版本与项目依赖配套，复现实例时按压缩包中的版本安装。

## 四、请求并渲染频道选项

频道数据通过接口获取，再渲染为 Select 选项：

```jsx
const [channels, setChannels] = useState([])

useEffect(() => {
  let active = true
  getChannelAPI().then(res => {
    if (active) setChannels(res.data.channels ?? [])
  })
  return () => { active = false }
}, [])

<Form.Item name="channel_id" label="频道">
  <Select placeholder="请选择文章频道">
    {channels.map(channel => (
      <Select.Option key={channel.id} value={channel.id}>
        {channel.name}
      </Select.Option>
    ))}
  </Select>
</Form.Item>
```

`value` 使用频道 id，提交表单时得到的 `channel_id` 才符合接口需要。频道数据在发布和列表页都会用到，后面可以封装为 `useChannel` 自定义 Hook。

## 五、先完成无图文章发布

在封面功能加入前，可以先用无图模式打通接口：

```jsx
const onFinish = async values => {
  const data = {
    title: values.title,
    channel_id: values.channel_id,
    content: values.content,
    cover: { type: 0, images: [] },
  }

  await createArticleAPI(data)
  message.success('发布成功')
}
```

随堂接口使用 `POST /mp/articles?draft=false`。先确认标题、频道、内容能够正确提交，再增加封面，能更快定位后续上传问题。请求体中的封面类型与 `images` 数量要一致。

![文章发布流程](assets/day08/17.png)

## 六、`Upload` 组件与上传结果

Ant Design `Upload` 通过 `action` 指定上传接口，`name` 指定文件字段名，`onChange` 取得文件列表：

```jsx
const [imageList, setImageList] = useState([])

<Upload
  name="image"
  action="http://geek.itheima.net/v1_0/upload"
  listType="picture-card"
  fileList={imageList}
  onChange={info => setImageList(info.fileList)}
>
  <PlusOutlined />
</Upload>
```

当传入 `fileList` 时，上传列表由组件状态控制。文件会经历 `uploading`、`done`、`error` 等状态；提交文章前应确认所有选中图片都成功上传。真正提交给文章接口的是上传接口返回的 URL，而不是浏览器中的 File 对象。

![封面上传](assets/day08/18.png)

## 七、无图、单图和三图切换

封面模式分别用 0、1、3 表示：

```jsx
const [imageType, setImageType] = useState(0)

<Radio.Group
  value={imageType}
  onChange={event => setImageType(event.target.value)}
>
  <Radio value={0}>无图</Radio>
  <Radio value={1}>单图</Radio>
  <Radio value={3}>三图</Radio>
</Radio.Group>

{imageType > 0 && (
  <Upload
    name="image"
    action="http://geek.itheima.net/v1_0/upload"
    listType="picture-card"
    fileList={imageList}
    maxCount={imageType}
    multiple={imageType > 1}
    onChange={info => setImageList(info.fileList)}
  >
    <PlusOutlined />
  </Upload>
)}
```

无图模式不显示上传区；单图最多 1 张；三图最多 3 张。`maxCount` 限制可见上传数量，但提交时仍要验证 `imageList.length === imageType`。

![封面模式](assets/day08/19.png)

## 八、在模式切换时暂存图片

若三图模式已经上传三张，切到单图只显示第一张，再切回三图希望恢复原来的三张，可用 `useRef` 保存完整列表：

```jsx
const cacheImageList = useRef([])
const [imageList, setImageList] = useState([])

const onUploadChange = info => {
  cacheImageList.current = info.fileList
  setImageList(info.fileList.slice(0, imageType))
}

const onTypeChange = event => {
  const type = event.target.value
  setImageType(type)
  setImageList(type === 0 ? [] : cacheImageList.current.slice(0, type))
}
```

`useRef` 的值改变不会单独触发渲染，适合保存切换时不直接展示的完整上传列表；真正显示哪几张仍由 state 中的 `imageList` 决定。注意删除图片时也要同步缓存，避免切换模式后被删除的图片又出现。

## 九、提交带封面的文章

```jsx
const onFinish = async values => {
  if (imageList.length !== imageType) {
    message.warning('封面类型和图片数量不匹配')
    return
  }
  const images = imageList.map(file => file.response?.data?.url)
  if (images.some(url => !url)) {
    message.warning('请等待图片上传完成')
    return
  }

  const data = {
    title: values.title,
    channel_id: values.channel_id,
    content: values.content,
    cover: { type: imageType, images },
  }
  await createArticleAPI(data)
  message.success('发布成功')
}
```

课程强调两层校验：模式与图片数量一致、上传结果 URL 已准备好。封面类型属于 `cover.type`，图片地址属于 `cover.images`。从 UI 状态转换到 API 请求体时，应把这个结构一次组装清楚。

## 十、文章列表：筛选区和表格结构

文章列表页面由上下两张 Card 组成：上方是状态、频道、日期筛选表单；下方是文章表格。

```jsx
<Card title="文章筛选">
  <Form initialValues={{ status: '' }}>
    <Form.Item name="status" label="状态">
      <Radio.Group>
        <Radio value="">全部</Radio>
        <Radio value={1}>待审核</Radio>
        <Radio value={2}>审核通过</Radio>
      </Radio.Group>
    </Form.Item>
    <Form.Item name="channel_id" label="频道">
      <Select placeholder="请选择频道" />
    </Form.Item>
    <Form.Item name="date" label="日期">
      <DatePicker.RangePicker />
    </Form.Item>
  </Form>
</Card>

<Card title="文章列表">
  <Table rowKey="id" columns={columns} dataSource={article.list} />
</Card>
```

第 08 天先把结构和真实列表数据展示出来，筛选提交、分页和删除在第 09 天完成。

![列表结构](assets/day08/20.png)

## 十一、复用频道数据：`useChannel`

发布页和文章列表筛选区都要频道下拉框，可以封装自定义 Hook：

```jsx
function useChannel() {
  const [channelList, setChannelList] = useState([])

  useEffect(() => {
    let active = true
    getChannelAPI().then(res => {
      if (active) setChannelList(res.data.channels ?? [])
    })
    return () => { active = false }
  }, [])

  return { channelList }
}
```

```jsx
const { channelList } = useChannel()
```

自定义 Hook 复用的是请求和状态逻辑；每个组件调用后拥有自己的状态实例。若项目希望多个页面完全共享一次请求结果，后续再考虑缓存或全局数据层。

## 十二、从接口数据渲染 Table

```jsx
const [article, setArticle] = useState({ list: [], count: 0 })

useEffect(() => {
  async function load() {
    const res = await getArticleListAPI({ page: 1, per_page: 4 })
    setArticle({
      list: res.data.results ?? [],
      count: res.data.total_count ?? 0,
    })
  }
  load()
}, [])
```

表格列描述如何从每一条文章记录中读取字段：

```jsx
const columns = [
  { title: '标题', dataIndex: 'title' },
  { title: '发布时间', dataIndex: 'pubdate' },
  { title: '阅读数', dataIndex: 'read_count' },
  {
    title: '封面',
    dataIndex: 'cover',
    render: cover => <img src={cover?.images?.[0] || img404} width={80} alt="" />,
  },
]
```

`rowKey="id" 为动态行提供稳定标识；无封面时使用占位图，避免 `<img>` 地址为空。

## 十三、文章状态 Tag

不同文章状态可以映射成不同 Tag，而不是在 JSX 中堆叠多层三元表达式：

```jsx
const statusView = {
  0: <Tag>草稿</Tag>,
  1: <Tag color="warning">待审核</Tag>,
  2: <Tag color="success">审核通过</Tag>,
  3: <Tag color="error">审核失败</Tag>,
}

const columns = [
  {
    title: '状态',
    dataIndex: 'status',
    render: value => statusView[value] ?? <Tag>未知状态</Tag>,
  },
]
```

状态值及含义应以接口文档为准；这里展示枚举渲染的写法。遇到未识别状态时给出兜底显示，避免单元格空白。

![文章状态](assets/day08/21.png)

## 十四、容易踩的坑与本日总结

- `Form.Item` 缺少 `name`，提交时拿不到对应字段。
- 频道 Select 的 `value` 误用名称，接口实际需要频道 id。
- 上传尚未完成就发布，`file.response` 为 `undefined`。
- 只设 `maxCount` 而不校验最终数量，单图/三图模式可能与请求体不符。
- `Upload` 受控后忘记在 `onChange` 中更新 `fileList`，图片无法正常显示。
- `useRef` 缓存未随删除同步，模式切换后旧图片重新出现。
- 文章列表的 `rowKey` 用数组下标，后续分页/删除会产生复用问题。

本日完成了极客头条的文章发布和基础列表展示。第 09 天继续把列表筛选、分页、删除、编辑回填及构建优化做成闭环。